# Python para Redes Urbanas

**Primeras Jornadas de Innovación Geoespacial · UTEC · Minas, Lavalleja · 16 de octubre de 2026**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FelipeSBarros/taller-sig-python-utec-2026/blob/main/T2_Python_Redes_Urbanas.ipynb)

En el taller anterior trabajamos con puntos, líneas, polígonos y ráster. Hoy vamos a mirar la ciudad de otra manera: como una **red de calles**, es decir, como un **grafo**. Con Python, [OSMnx](https://osmnx.readthedocs.io/) y los datos de OpenStreetMap vamos a calcular rutas, entender cómo funciona el algoritmo que las encuentra, analizar la orientación de las calles y ponderar los recorridos con datos espaciales, como la altitud. Así, la pregunta por el mejor camino pasa a ser un problema de análisis espacial y de toma de decisiones.

Material producido por [Felipe Sodré Mendes Barros](https://felipesbarros.github.io/).

### Instalando y cargando las librerías (módulos de Python)

* [OSMnx](https://osmnx.readthedocs.io/): descargar, analizar y visualizar redes de calles de OpenStreetMap;
* [geopandas](https://geopandas.org/), [rasterio](https://rasterio.readthedocs.io/) y [matplotlib](https://matplotlib.org/): los mismos del taller anterior, para vectores, ráster y gráficos.

En Colab casi todo ya viene instalado: solo hace falta instalar `osmnx`.

In [ ]:
%pip install osmnx --quiet

In [ ]:
import os
import urllib.request

import geopandas as gpd
import matplotlib.pyplot as plt
import osmnx as ox
import pandas as pd
import rasterio

from rasterio.plot import show

### Los datos

La red de calles la vamos a descargar en vivo desde OpenStreetMap. Los demás datos (el modelo digital del terreno del taller anterior y una copia de respaldo de la red de Minas) están en el repositorio [taller-sig-python-utec-2026](https://github.com/FelipeSBarros/taller-sig-python-utec-2026).

In [ ]:
URL = "https://raw.githubusercontent.com/FelipeSBarros/taller-sig-python-utec-2026/main/datos/"
LUGAR = "Minas, Lavalleja, Uruguay"

def descargar(archivo):
    # baja un archivo del repositorio del taller (si todavía no está) y devuelve su nombre
    if not os.path.exists(archivo):
        urllib.request.urlretrieve(URL + archivo, archivo)
    return archivo

# Grafos

Un grafo está formado por un conjunto de **nodos** (o vértices) y de **aristas** (o enlaces) que los conectan. En el fondo, para nosotros, son **puntos** y **líneas**: los mismos datos vectoriales del taller anterior, pero organizados de otra manera, porque ahora lo importante es **quién está conectado con quién**. En una red de calles, los nodos son las intersecciones (y los finales de calle) y las aristas son los tramos de calle entre ellas.

Además de nodos y aristas, cada uno de esos elementos puede tener **atributos** (el nombre de la calle, su largo, la velocidad máxima, la altitud de una esquina...) y las aristas pueden tener **dirección**: una calle de mano única va de un nodo a otro, pero no al revés.

### Un poco de historia: los puentes de Königsberg

En 1736, el matemático Leonhard Euler se preguntó si era posible recorrer la ciudad de Königsberg cruzando cada uno de sus siete puentes una sola vez. Para responder, dejó de lado el mapa y se quedó con lo esencial: cada zona de tierra pasó a ser un punto (un nodo) y cada puente, una línea entre dos puntos (una arista). Con ese dibujo demostró que el paseo era imposible, y así nació la **teoría de grafos**.

<figure>
<img width=50% src="https://upload.wikimedia.org/wikipedia/commons/5/5d/Konigsberg_bridges.png">
<figcaption>Los siete puentes de Königsberg. Fuente: <a href="https://commons.wikimedia.org/wiki/File:Konigsberg_bridges.png">Wikimedia Commons</a></figcaption>
</figure>

Esa misma idea (quedarse con las conexiones y olvidarse del resto) es la que usan hoy los navegadores del celular, el ruteo de los camiones de reparto o el estudio de la accesibilidad a escuelas y hospitales. En [Wikipedia](https://es.wikipedia.org/wiki/Teor%C3%ADa_de_grafos) hay más sobre teoría de grafos y, si te interesa el potencial del análisis de redes de calles, el artículo de [`OSMnx`](https://arxiv.org/pdf/1611.01890.pdf), la librería que vamos a usar, es una muy buena puerta de entrada.

### OpenStreetMap

OpenStreetMap (OSM) es un proyecto de mapeo colaborativo para crear un mapa (en realidad, una base de datos espaciales) libre y editable del mundo, inspirado en sitios como Wikipedia.

Los datos los desarrolla y mantiene con rigor una **comunidad** voluntaria de mapeadores, que cargan y revisan datos de receptores GPS, fotografías aéreas, imágenes de satélite y otras fuentes libres. Los mapeadores, con su conocimiento local, editan el mapa con programas abiertos como iD o JOSM.

Todos los mapas, datos descriptivos y metadatos de OSM son datos abiertos, disponibles bajo la licencia Open Database License (ODbL).

**Vamos a aprovechar el trabajo de esa comunidad para explorar los datos de Minas.**

* [OpenStreetMap en Wikipedia](https://es.wikipedia.org/wiki/OpenStreetMap)
* [Sitio de OSM](https://www.openstreetmap.org/)

## Descargando la red de calles como grafo

El módulo `OSMnx` (que importamos con el alias `ox`) fue desarrollado para acceder, manipular y analizar datos geográficos con estructura de grafo.

Para descargar la red usamos la función `graph_from_place()`, a la que le pasamos el lugar de interés como texto. 

In [ ]:
G = ox.graph_from_place(LUGAR, network_type="drive")
G

**Celda de rescate:** si la descarga falla o tarda más de dos minutos, interrumpila (botón ■) y ejecutá esta celda, que carga una copia de la misma red guardada en el repositorio del taller.

In [ ]:
G = ox.load_graphml(descargar("grafo_minas_drive.graphml"))
G

¿Qué es este objeto? Un `MultiDiGraph` de `networkx`: un grafo **dirigido** (*Di*, de *directed*: las aristas tienen sentido, como las calles de mano única) que admite **más de una arista** entre los mismos dos nodos (*Multi*). Veamos cuántos nodos y aristas tiene la red de Minas:

In [ ]:
print(len(G.nodes), "nodos")
print(len(G.edges), "aristas")

### Visualizando el grafo

Para dibujarlo usamos la función [`plot_graph()`](https://osmnx.readthedocs.io/en/stable/user-reference.html#osmnx.plot.plot_graph) de `OSMnx`, a la que le pasamos el grafo descargado.

In [ ]:
ox.plot_graph(G)

Como en el taller anterior, podemos cambiar cómo se dibuja cada elemento:

* `bgcolor`: el color del fondo;
* `node_color` y `node_size`: el color y el tamaño de los nodos (con `node_size=0` desaparecen);
* `edge_color` y `edge_linewidth`: el color y el grosor de las aristas;
* `figsize`: el tamaño de la figura, por ejemplo `figsize=(10, 10)`.

In [ ]:
# Ejercicio: dibujá el grafo con fondo blanco (bgcolor="white"), aristas negras y nodos rojos más chicos.


In [ ]:
# Ejercicio: ahora probá sin nodos (node_size=0), con aristas grises más finas (edge_linewidth=0.5)
# y del tamaño que quieras. ¿Qué parte de la ciudad tiene más calles?


## Algunos parámetros

Con `network_type` elegimos qué red queremos: `"drive"` (para autos), `"walk"` (a pie), `"bike"` (en bicicleta) o `"all"` (todas). Empecemos por la red para autos de Minas.

`OSMnx` tiene otras funciones para descargar grafos, como `graph_from_address()`, `graph_from_point()` o `graph_from_bbox()`, [entre otras](https://osmnx.readthedocs.io/en/stable/user-reference.html#module-osmnx.graph).

La descarga puede tardar uno o dos minutos: la consulta va a los servidores de OpenStreetMap, que estamos usando todos al mismo tiempo.

## Entendiendo el grafo

Como dijimos al principio, todo grafo está formado por **nodos** (`nodes`) y **aristas** (`edges`), y cada uno tiene sus atributos.

### Nodos

Cada nodo tiene un identificador (el mismo que tiene en OpenStreetMap) y, como atributos, sus coordenadas (`x` = longitud, `y` = latitud) y la cantidad de calles que llegan a él (`street_count`).

In [ ]:
# los primeros 5 nodos, con sus atributos
list(G.nodes(data=True))[:5]

### Aristas

Una arista se identifica por el nodo de origen, el nodo de destino y un número (`0`, `1`...) que distingue las aristas que unen los mismos dos nodos. Entre sus atributos están el nombre de la calle (`name`), el tipo de vía (`highway`), si es de mano única (`oneway`) y el largo en metros (`length`). Ese largo va a ser muy importante en un rato.

In [ ]:
# las primeras 3 aristas, con sus atributos
list(G.edges(keys=True, data=True))[:3]

### Del grafo a los datos vectoriales

¿Nodos y aristas no son, al final, puntos y líneas? Sí: con `ox.graph_to_gdfs()` convertimos el grafo en **dos** `GeoDataFrame`, uno de nodos (puntos) y otro de aristas (líneas), y volvemos a todo lo que vimos en el taller anterior: tablas de atributos, `.plot()`, colores por columna...

In [ ]:
nodos, aristas = ox.graph_to_gdfs(G)
aristas.head()

In [ ]:
# Ejercicio: dibujá las aristas coloreadas según si la calle es de mano única (column="oneway"), con leyenda.
# Pista: es igual que el mapa de focos coloreado por "confianza" del taller anterior.


Vale la pena guardar esta idea: **el grafo y la capa de líneas son el mismo dato**. La diferencia es que el grafo, además, sabe qué tramos se tocan y en qué sentido se puede circular por ellos. Esa es la información que necesitamos para responder la pregunta que sigue.

---
# ¿Qué hacemos con un grafo?

Por ejemplo, **calcular rutas**: dado un punto de partida y uno de llegada, encontrar el camino más corto entre los dos por las calles de la ciudad.

`OSMnx` tiene la función `shortest_path()`, que recibe el grafo, el **nodo** de origen y el **nodo** de destino. Pero acá aparece un problema:

> ¿Qué hago si **solo** tengo la dirección de dónde estoy y de adónde quiero ir, y no sé las coordenadas, ni mucho menos el identificador de un nodo?

La respuesta tiene dos pasos: **geocodificar** la dirección (pasar del texto a coordenadas) y buscar el **nodo más cercano** a esas coordenadas.

### Geocodificación

> Un *geocode* es la representación de un objeto, fenómeno, lugar o paisaje a partir de un par de coordenadas.

La geocodificación (*geocoding* o *address geocoding*) es el proceso de encontrar el mejor *geocode* disponible para una dirección.

**No hay magia en esto: se trata de una consulta a una base de datos.** En nuestro caso, la base de datos de OpenStreetMap (con su servicio [Nominatim](https://nominatim.openstreetmap.org/)), aunque Google y otras empresas también ofrecen el servicio. Y por eso mismo, **la precisión de la geocodificación es relativa**: depende de lo que esté cargado en esa base de datos.

`OSMnx` tiene la función [`geocode()`](https://osmnx.readthedocs.io/en/stable/user-reference.html#osmnx.geocoder.geocode), a la que le pasamos la dirección o el nombre del lugar. Hagamos la geocodificación de nuestro punto de partida: la **Terminal de Ómnibus de Minas**.

**Prestá mucha atención a [lo que devuelve esta función](https://osmnx.readthedocs.io/en/stable/user-reference.html#osmnx.geocoder.geocode).**

In [ ]:
terminal = ox.geocode("Terminal de Ómnibus, Minas, Uruguay")
terminal

In [ ]:
# ¿lon/lat o lat/lon?
print("latitud  =", terminal[0])
print("longitud =", terminal[1])

Ojo: `geocode()` devuelve **(latitud, longitud)**, al revés del orden (x, y) = (longitud, latitud) que veníamos usando. Es una fuente clásica de errores.

**La precisión es relativa:** mirá qué pasa si buscamos "Parque Rodó" sin decir en qué ciudad.

In [ ]:
print("Parque Rodó:                ", ox.geocode("Parque Rodó"))
print("Parque Rodó, Minas, Uruguay:", ox.geocode("Parque Rodó, Minas, Uruguay"))

El primero cayó en Montevideo, a más de 100 km. La base de datos nos devolvió el "Parque Rodó" más importante que encontró, que no era el que queríamos. Por eso conviene escribir la dirección completa, con la ciudad y el país.

### Encontrando el nodo más cercano

Muy bien, ya tenemos coordenadas. Pero necesitamos saber dónde cae esa coordenada dentro de nuestro grafo, o mejor dicho: **cuál es el nodo más cercano a esa coordenada**.

Para eso usamos [`ox.nearest_nodes()`](https://osmnx.readthedocs.io/en/stable/user-reference.html#osmnx.distance.nearest_nodes), a la que le pasamos el grafo, la **longitud** (X) y la **latitud** (Y). Fijate en el orden: como `terminal` es (lat, lon), la longitud es `terminal[1]` y la latitud, `terminal[0]`.

In [ ]:
nodo_terminal = ox.nearest_nodes(G, terminal[1], terminal[0])
nodo_terminal

### Repitiendo el proceso para el destino

Perfecto: ya tenemos el nodo de origen. Ahora necesitamos el de destino. Como siempre, no lo sabemos: solo sabemos que queremos ir al **Parque Zorrilla**. Vamos a usar este mismo origen y destino en todo el taller, para poder comparar los resultados. Hagamos:

1. la geocodificación,
2. la búsqueda del nodo más cercano y, por fin,
3. el cálculo del camino más corto.

In [ ]:
# Ejercicio: geocodificá "Parque Zorrilla, Minas, Uruguay" y guardalo en un objeto llamado zorrilla.


In [ ]:
# Ejercicio: buscá el nodo más cercano al parque y guardalo en nodo_zorrilla.
# Pista: ox.nearest_nodes(G, longitud, latitud)


## Calculando el camino más corto

Ahora que tenemos el nodo de origen y el de destino, podemos calcular el camino más corto entre los dos con [`ox.shortest_path()`](https://osmnx.readthedocs.io/en/stable/user-reference.html#osmnx.routing.shortest_path), indicando el grafo, el nodo de origen y el nodo de destino.

In [ ]:
# Ejercicio: calculá el camino más corto entre nodo_terminal y nodo_zorrilla y guardalo en ruta.


El resultado es una **lista con los identificadores de los nodos** por los que pasa el camino, desde el origen hasta el destino.

### Dibujando la ruta

Para verla usamos `ox.plot_graph_route()`, a la que le pasamos el grafo y el resultado de `shortest_path()`. Acepta los mismos parámetros de `plot_graph()` (`bgcolor`, `node_size`, `edge_color`...) y, además, `route_color` para el color de la ruta.

In [ ]:
# Ejercicio: dibujá la ruta con ox.plot_graph_route(G, ruta), con fondo blanco, sin nodos
# y con la ruta en azul (route_color="blue").


**Para pensar:** ¿la ruta que dibujó el algoritmo es la que habrías hecho vos? Si conocés Minas, ¿hay algo raro? Acordate de que estamos en la red para **autos**: el grafo es dirigido y respeta las calles de mano única, así que la vuelta del Parque Zorrilla a la Terminal no es el mismo camino que la ida (2.560 m a la vuelta, contra 2.566 m a la ida).

---
# ¿Cómo funciona el algoritmo?

El `shortest_path()` puede parecer **mágico**, pero no lo es. Lo que hace es buscar, entre todos los caminos posibles, el que **minimiza un costo**. A cada arista le corresponde un valor (su **peso**) y el costo de un camino es la suma de los pesos de las aristas por las que pasa. Por defecto, el peso es el largo de la calle (`length`), así que el camino de menor costo es el más corto.

Para que se entienda, miremos un grafo de juguete: cinco lugares de Minas y las calles que los unen, con el largo de cada una medido en **cuadras**. ¿Cuál es el camino más corto de la Terminal al Parque? Pensalo un momento antes de seguir.

<img width=70% src="https://raw.githubusercontent.com/FelipeSBarros/taller-sig-python-utec-2026/main/img/grafo_juguete_1.png">

El algoritmo que usa `OSMnx` por detrás se llama [**algoritmo de Dijkstra**](https://es.wikipedia.org/wiki/Algoritmo_de_Dijkstra) (por Edsger Dijkstra, que lo publicó en 1959). En la documentación se puede ver cómo lo usan [`ox.shortest_path()`](https://osmnx.readthedocs.io/en/stable/user-reference.html#osmnx.routing.shortest_path) y la función de `networkx` que hace el trabajo, [`dijkstra_path()`](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.shortest_paths.weighted.dijkstra_path.html). La idea es así:

1. Arranca en el origen, con costo 0. Todos los demás nodos empiezan con costo "infinito" (todavía no sabemos cómo llegar).
2. Mira los vecinos del nodo en el que está y anota, para cada uno, el costo de llegar pasando por ahí (el costo acumulado + el peso de la arista), si es menor que el que ya tenía anotado.
3. De todos los nodos que todavía no visitó, pasa al que tiene el **menor costo anotado**: ese costo ya es definitivo, porque cualquier otro camino hasta él sería más largo.
4. Repite los pasos 2 y 3 hasta llegar al destino. Para reconstruir el camino, alcanza con recordar desde qué nodo llegó a cada uno.

En nuestro ejemplo: desde la Terminal, el Estadio queda a 2 cuadras y la Plaza a 4. Visita el Estadio (2): desde ahí la Plaza queda a 2 + 1 = 3, que es menos que 4, así que corrige el valor; y el Teatro, a 2 + 8 = 10. Visita la Plaza (3): el Teatro queda a 3 + 5 = 8, que mejora 10, y el Parque a 3 + 10 = 13. Visita el Teatro (8): el Parque queda a 8 + 3 = 11, que mejora 13. Llega al Parque con costo 11, pasando por el Estadio, la Plaza y el Teatro.

<figure>
<img width=35% src="https://upload.wikimedia.org/wikipedia/commons/5/57/Dijkstra_Animation.gif">
<figcaption>El algoritmo de Dijkstra en acción. Fuente: <a href="https://commons.wikimedia.org/wiki/File:Dijkstra_Animation.gif">Wikimedia Commons</a></figcaption>
</figure>

Ahora imaginá que hay una obra en la calle entre el Estadio y la Plaza, y para pasar hay que dar una vuelta de 5 cuadras. ¿Cambia el camino? Antes de mirar la figura, repetí los pasos del algoritmo con el nuevo peso.

<img width=70% src="https://raw.githubusercontent.com/FelipeSBarros/taller-sig-python-utec-2026/main/img/grafo_juguete_2.png">

El algoritmo no "sabe" nada de calles ni de obras: solo suma pesos y se queda con el camino más barato. **Si cambiamos los pesos, cambia el camino.**

En el grafo de Minas pasa exactamente lo mismo. Cada arista tiene su `length` (en metros) y eso es lo que `ox.shortest_path()` minimiza por defecto (`weight="length"`).

---
# ¿Y si cambio el campo?

Si el peso es lo que define el camino, podemos usar **otro atributo** como peso. Por ejemplo, el **tiempo** de viaje en lugar de la distancia: no siempre el camino más corto es el más rápido.

`OSMnx` puede calcular dos atributos nuevos para cada arista:

* [`ox.add_edge_speeds()`](https://osmnx.readthedocs.io/en/stable/user-reference.html#osmnx.routing.add_edge_speeds): la velocidad (`speed_kph`, en km/h). Usa la velocidad máxima cargada en OpenStreetMap (`maxspeed`) y, cuando falta, la estima con el promedio de las calles del mismo tipo (`highway`).
* [`ox.add_edge_travel_times()`](https://osmnx.readthedocs.io/en/stable/user-reference.html#osmnx.routing.add_edge_travel_times): el tiempo de viaje (`travel_time`, en segundos) = largo / velocidad.

In [ ]:
G = ox.add_edge_speeds(G)
G = ox.add_edge_travel_times(G)

nodos, aristas = ox.graph_to_gdfs(G)
aristas[["name", "highway", "maxspeed", "speed_kph", "length", "travel_time"]].head(10)

¿En cuántas calles de Minas está cargada la velocidad máxima? Esto también es parte de leer el dato: si falta mucho, las velocidades son, en su mayoría, estimadas. Veamos también el mapa de velocidades: ¿cuáles son las calles "rápidas"?

In [ ]:
print(f"Aristas con maxspeed: {aristas['maxspeed'].notna().mean():.0%}")
ax = aristas.plot(column="speed_kph", legend=True, cmap="viridis", figsize=(10, 10))
ax.set_title("Velocidad de cada calle (km/h)")
ax.set_axis_off()

Ahora calculamos dos rutas entre la **Terminal** y el **Parque Zorrilla** (los mismos nodos de antes): una que minimiza la distancia (`weight="length"`) y otra que minimiza el tiempo (`weight="travel_time"`).

Para dibujarlas juntas usamos [`ox.plot_graph_routes()`](https://osmnx.readthedocs.io/en/stable/user-reference.html#osmnx.plot.plot_graph_routes) (fijate la **s** del final), que recibe las rutas en forma de **lista**: en Python, una lista se escribe entre corchetes, `[ruta_distancia, ruta_tiempo]`. Con `route_colors` le damos un color a cada ruta, también como lista.

In [ ]:
ruta_distancia = ox.shortest_path(G, nodo_terminal, nodo_zorrilla, weight="length")
ruta_tiempo = ox.shortest_path(G, nodo_terminal, nodo_zorrilla, weight="travel_time")

ox.plot_graph_routes(G, [ruta_distancia, ruta_tiempo], route_colors=["red", "blue"],
                     bgcolor="white", node_size=0, edge_color="grey", edge_linewidth=0.5)

En rojo, la más corta; en azul, la más rápida. Son dos caminos distintos, aunque en este caso casi empatan: la ruta por tiempo es apenas unos metros más larga y las dos tardan prácticamente lo mismo (alrededor de 3 minutos y medio). Cambiar el peso cambia el camino, aunque el costo final sea parecido.

**Para pensar:**

* ¿Qué tan confiables son esos tiempos, si la mayoría de las velocidades son estimadas?
* ¿Qué no está considerando el modelo (semáforos, tránsito, pendientes)?
* La pregunta "¿cuál es el mejor camino?" no tiene una única respuesta: **depende del criterio** que elijamos como peso.

---
# **Pausa** (?)
---

### Celda de rescate

Si algo de lo anterior no funcionó, ejecutá esta celda: carga lo que necesitamos para seguir.

In [ ]:
URL = "https://raw.githubusercontent.com/FelipeSBarros/taller-sig-python-utec-2026/main/datos/"
LUGAR = "Minas, Lavalleja, Uruguay"

def descargar(archivo):
    if not os.path.exists(archivo):
        urllib.request.urlretrieve(URL + archivo, archivo)
    return archivo

G = ox.load_graphml(descargar("grafo_minas_drive.graphml"))

# Ponderando las rutas con la altitud

Minas está rodeada de sierras y tiene lomas dentro de la ciudad. Si vamos **a pie** (o en bicicleta), el camino más corto puede obligarnos a subir una loma que preferiríamos evitar. ¿Podemos pedirle al algoritmo que "evite lo alto"?

Sí: ya vimos que alcanza con cambiar el peso. Pero el grafo de OpenStreetMap no trae la altitud. La vamos a sacar del **modelo digital del terreno (MDT)** de Minas que usamos en el taller anterior (IDE Uruguay, 5 m) y la vamos a sumar a cada nodo con una consulta espacial: para cada nodo, el valor del píxel del ráster donde cae.

Primero, la red **a pie** de Minas:

In [ ]:
G_pie = ox.graph_from_place(LUGAR, network_type="walk")
print(len(G_pie.nodes), "nodos,", len(G_pie.edges), "aristas")

**Celda de rescate:** si la descarga falla o tarda demasiado, interrumpila y ejecutá esta:

In [ ]:
G_pie = ox.load_graphml(descargar("grafo_minas_walk.graphml"))

Ahora el MDT. `OSMnx` necesita el archivo en la computadora (en Colab, en el panel de Archivos), así que primero lo descargamos con nuestra función `descargar()` y después lo abrimos con `rasterio`, como en el taller anterior.

In [ ]:
mdt = rasterio.open(descargar("mdt_minas.tif"))
print(mdt.crs)
show(mdt, cmap="terrain")

### El CRS, otra vez

El grafo está en coordenadas geográficas (EPSG:4326) y el MDT en coordenadas proyectadas (EPSG:5382, en metros). Para cruzarlos tienen que estar en el **mismo** sistema de referencia. Como en el taller anterior, llevamos el vector al CRS del ráster: `ox.project_graph()` reproyecta el grafo (es el `to_crs()` de los grafos).

Después, `ox.elevation.add_node_elevations_raster()` le agrega a cada nodo el atributo `elevation` con el valor del MDT.

In [ ]:
G_pie = ox.project_graph(G_pie, to_crs=mdt.crs)
G_pie = ox.elevation.add_node_elevations_raster(G_pie, "mdt_minas.tif")

# un nodo cualquiera, ahora con su altitud
list(G_pie.nodes(data=True))[0]

Dibujemos los nodos coloreados por altitud. `ox.plot.get_node_colors_by_attr()` arma un color para cada nodo según un atributo:

In [ ]:
colores = ox.plot.get_node_colors_by_attr(G_pie, "elevation", cmap="terrain")
ox.plot_graph(G_pie, node_color=colores, node_size=10, edge_color="lightgrey", edge_linewidth=0.5, bgcolor="white")

### Un peso que penaliza la altitud

A cada arista le damos la altitud media de sus dos nodos, la pasamos a una escala de 0 (la esquina más baja de Minas) a 1 (la más alta) y calculamos un costo:

`costo = largo × (1 + α · altitud normalizada)`

* Con **α = 0**, el costo es el largo: volvemos a la ruta más corta.
* Cuanto mayor es **α**, más "cara" resulta una calle alta, y el algoritmo prefiere dar una vuelta más larga por abajo.

α funciona como una **perilla**: de "llegar rápido" a "no subir".

In [ ]:
altitudes = [d["elevation"] for n, d in G_pie.nodes(data=True)]
alt_min, alt_max = min(altitudes), max(altitudes)
print(f"Altitud de las esquinas de Minas: de {alt_min:.0f} a {alt_max:.0f} m")

for u, v, k, d in G_pie.edges(keys=True, data=True):
    altitud_media = (G_pie.nodes[u]["elevation"] + G_pie.nodes[v]["elevation"]) / 2
    d["altitud_norm"] = (altitud_media - alt_min) / (alt_max - alt_min)

def agregar_costo(G, alfa):
    for u, v, k, d in G.edges(keys=True, data=True):
        d["costo"] = d["length"] * (1 + alfa * d["altitud_norm"])

Vamos a ir de la **Terminal de Ómnibus** al **Parque Zorrilla**, dos lugares en zonas bajas separados por la loma del centro. Como el grafo ahora está en metros (EPSG:5382), las coordenadas de la geocodificación también tienen que pasar a ese CRS antes de buscar el nodo más cercano. Lo resolvemos con una pequeña función que hace los tres pasos:

In [ ]:
def nodo_cercano(G, lugar):
    lat, lon = ox.geocode(lugar)
    punto = gpd.GeoSeries(gpd.points_from_xy([lon], [lat]), crs="EPSG:4326").to_crs(G.graph["crs"])
    return ox.nearest_nodes(G, punto.x.iloc[0], punto.y.iloc[0])

origen = nodo_cercano(G_pie, "Terminal de Ómnibus, Minas, Uruguay")
destino = nodo_cercano(G_pie, "Parque Zorrilla, Minas, Uruguay")

Calculamos la ruta con tres valores de α y resumimos cada una: el largo, la altitud máxima que alcanza y la **subida acumulada** (la suma de todos los tramos en subida).

In [ ]:
def resumen(G, ruta):
    t = ox.routing.route_to_gdf(G, ruta)
    alts = [G.nodes[n]["elevation"] for n in ruta]
    subida = sum(max(0, b - a) for a, b in zip(alts[:-1], alts[1:]))
    return {"largo (m)": round(t["length"].sum()), "altitud máx. (m)": round(max(alts)),
            "subida acumulada (m)": round(subida)}

rutas_alt = {}
tabla = []
for alfa in [0, 3, 10]:
    agregar_costo(G_pie, alfa)
    rutas_alt[alfa] = ox.shortest_path(G_pie, origen, destino, weight="costo")
    tabla.append({"α": alfa, **resumen(G_pie, rutas_alt[alfa])})
pd.DataFrame(tabla)

Y las dibujamos sobre el MDT, como en el taller anterior: primero el ráster, después las calles y, encima, las rutas.

In [ ]:
nodos_pie, aristas_pie = ox.graph_to_gdfs(G_pie)
colores_rutas = {0: "red", 3: "blue", 10: "black"}

fig, ax = plt.subplots(figsize=(10, 10))
show(mdt, ax=ax, cmap="terrain", alpha=0.8)
aristas_pie.plot(ax=ax, color="grey", linewidth=0.3)
for alfa, r in rutas_alt.items():
    ox.routing.route_to_gdf(G_pie, r).plot(ax=ax, color=colores_rutas[alfa], linewidth=3, label=f"α = {alfa}")
xmin, ymin, xmax, ymax = nodos_pie.total_bounds
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.legend()
ax.set_title("Terminal → Parque Zorrilla: ruta más corta y rutas que evitan lo alto")
ax.set_axis_off()

**Para pensar:**

* ¿Vale la pena caminar casi un kilómetro más para subir menos? ¿Para quién sí (una persona mayor, alguien con un carrito de bebé, en bicicleta) y para quién no?
* El par origen–destino importa: si el destino está en lo alto (por ejemplo, el Cerro Artigas), no hay forma de evitar la subida y las rutas casi no cambian.
* Acá penalizamos **estar alto**. Otra opción es penalizar **subir**: `ox.add_edge_grades()` calcula la pendiente de cada arista (la diferencia de altitud dividida por el largo) y se puede usar en el costo. Con cualquier otra capa espacial (NDVI para caminar por la sombra, zonas inundables, iluminación) la lógica es la misma: **un dato espacial que se convierte en peso.**

---
# Orientación de las calles

Volvamos a la red de autos. Además de rutas, el grafo permite describir la **forma** de la ciudad. Por ejemplo, ¿hacia dónde apuntan sus calles?

`ox.add_edge_bearings()` agrega a cada arista su orientación (`bearing`): el ángulo, en grados y en el sentido de las agujas del reloj, entre el norte y la línea que va del nodo de origen al de destino.

In [ ]:
G = ox.add_edge_bearings(G)
list(G.edges(data="bearing"))[:5]

Con esas orientaciones armamos un histograma **polar**: un gráfico circular que muestra con qué frecuencia aparece cada dirección.

Antes convertimos el grafo a **no dirigido** (`ox.convert.to_undirected()`): así, una calle de doble mano cuenta una sola vez, y cada calle se representa en las dos direcciones opuestas (norte y sur, por ejemplo), que es lo que nos interesa para describir la trama.

In [ ]:
G_nd = ox.convert.to_undirected(G)
ox.plot_orientation(G_nd, title="Minas")

El centro de Minas tiene una trama en **damero** (cuadrícula), como muchas ciudades de origen colonial en América: por eso se destacan cuatro direcciones, a 90° entre sí. Como la cuadrícula está girada respecto del norte, los picos aparecen en diagonal (noreste, sureste, suroeste y noroeste), y las barras más chicas corresponden a los barrios que crecieron fuera del damero. En ciudades que crecieron sin planificación, o siguiendo el relieve, el histograma se reparte en todas las direcciones. Geoff Boeing, el autor de `OSMnx`, comparó [la orientación de las calles de 100 ciudades del mundo](https://geoffboeing.com/2018/07/comparing-city-street-orientations/).

In [ ]:
# Ejercicio: descargá la red de otra ciudad de Uruguay (por ejemplo "Colonia del Sacramento, Uruguay"
# o "Treinta y Tres, Uruguay") y compará su orientación con la de Minas. Puede tardar un poco.
# Pasos: graph_from_place() → add_edge_bearings() → convert.to_undirected() → plot_orientation()


### Figure-ground

Los diagramas [*figure-ground*](https://geoffboeing.com/2017/03/urban-form-figure-ground/) son una herramienta clásica del urbanismo para representar la relación entre lo construido y lo no construido. Con `ox.plot_figure_ground()` dibujamos las calles de una milla cuadrada alrededor del centro de la red, con el ancho según el tipo de vía:

In [ ]:
ox.plot_figure_ground(G, color="black", bgcolor="white")

---
# Tu mapa (evaluación)

> **Entrega cerrada.** El plazo venció a las 23:59 del viernes 16 de octubre de 2026 y el formulario ya no acepta respuestas. El ejercicio queda como práctica.

Llegó el momento de hacer **tu propio mapa**, con lo que aprendiste hoy. ~~Es la entrega para la evaluación del taller.~~

**Consigna:**
1. Elegí un origen y un destino en Minas (o en otra ciudad) y calculá **al menos dos rutas** con criterios distintos: distancia y tiempo, o distintos valores de α.
2. Dibujalas en un mismo mapa, cada una con un color, sobre la red de calles (o sobre el MDT).
3. Ponele un **título** que incluya **tu nombre**, y una leyenda que diga qué es cada ruta.
4. Guardalo con `plt.savefig("apellido_nombre.png", dpi=150)` en la misma celda del mapa.
5. Descargalo: en el panel de **Archivos** de Colab (el ícono de carpeta, a la izquierda), hacé clic derecho sobre `apellido_nombre.png` y elegí **Descargar**.
6. ~~Subilo al formulario de entrega, junto con **2 o 3 líneas** que cuenten qué muestra el mapa y cuál de las rutas elegirías y por qué, **hasta las 23:59 del viernes 16 de octubre**~~

~~[https://forms.gle/AF5s89nZzDozbyVj6](https://forms.gle/AF5s89nZzDozbyVj6)~~

**Criterios:** que tenga al menos dos rutas con criterios distintos, título con tu nombre, colores y leyenda adecuados, y una interpretación breve y coherente.

In [ ]:
# Tu mapa: escribí aquí el código. Podés partir de cualquier celda anterior.
# fig, ax = plt.subplots(figsize=(10, 10))
# ...
# ax.set_title("... | Nombre Apellido")
# plt.savefig("apellido_nombre.png", dpi=150)


---
# Para cerrar: ¿cuál es el mejor camino?

Hoy vimos que el "camino más corto" no es una verdad única: es el resultado de un algoritmo que minimiza **el peso que nosotros elegimos**. Distancia, tiempo, altitud, sombra, seguridad... cada criterio es una decisión, y detrás de cada decisión hay personas con necesidades distintas. Por eso la pregunta por el mejor camino es, también, un problema de análisis espacial y de toma de decisiones.

### Para seguir aprendiendo

* [Introduction to urban accessibility](https://ipeagit.github.io/intro_access_book/) (IPEA): libro libre, en inglés y portugués, sobre cómo medir el acceso a oportunidades en la ciudad.
* [Documentación y ejemplos de OSMnx](https://github.com/gboeing/osmnx-examples).
* [Blog de Geoff Boeing](https://geoffboeing.com/): análisis de redes urbanas con datos abiertos.
* [OpenStreetMap Uruguay](https://www.openstreetmap.org/): si encontraste una calle mal cargada en Minas, podés corregirla vos.